# LLDP Flow-Based IDS - Complete ML Pipeline

Random Forest classifier for LLDP attack detection in SDN environments.

**References:**
- Flow-based IDS: github.com/arsheen/IDS-on-SDN-using-Machine-Learning
- Feature engineering: github.com/rana-uzair-ahmed/MininetIDS
- Random Forest optimization: scikit-learn.org/stable/modules/ensemble.html#random-forests
- Class imbalance: github.com/scikit-learn-contrib/imbalanced-learn

In [ ]:
# Upload FINAL_LLDP_DATASET_COMPLETE_enriched.csv to /content/ before running
# In Colab: Files icon (left sidebar) → Upload button
BASE_PATH = '/content'

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix, f1_score,
    accuracy_score, precision_recall_fscore_support
)
import joblib
import time
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
sns.set_style('whitegrid')

## 1. Load and Explore Dataset

In [ ]:
# From: github.com/arsheen/IDS-on-SDN-using-Machine-Learning (dataset loading)
df_raw = pd.read_csv(f'{BASE_PATH}/FINAL_LLDP_DATASET_COMPLETE_enriched.csv')

print(f"Dataset: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns")
print(f"\nColumns: {list(df_raw.columns)}")

# Class distribution
print(f"\nClass Distribution:")
print(df_raw['label'].value_counts())
print(f"\nImbalance Ratio: {df_raw['label'].value_counts().max() / df_raw['label'].value_counts().min():.2f}:1")

## 2. Feature Engineering & Cleaning

In [ ]:
# From: github.com/rana-uzair-ahmed/MininetIDS (feature selection)

df = df_raw.drop_duplicates()
print(f"After deduplication: {len(df):,} rows ({df_raw.shape[0] - len(df):,} removed)")

# Behavioral features for flow-based detection
BEHAVIORAL_FEATURES = [
    'packet_rate_inst', 'packet_rate_win', 'packet_rate',
    'burstiness_cv', 'size_z_src', 'time_delta',
    'age_since_first', 'tlv_density', 'ttl_dev',
    'ttl_anom_flag', 'is_lldp_mc'
]

# Keep only features present in dataset
available_features = [f for f in BEHAVIORAL_FEATURES if f in df.columns]
print(f"\nAvailable features ({len(available_features)}): {available_features}")

In [ ]:
# Remove high-missing features (>50% threshold)
features_to_keep = []
for feature in available_features:
    missing_pct = df[feature].isnull().sum() / len(df)
    if missing_pct <= 0.50:
        features_to_keep.append(feature)
        if missing_pct > 0:
            median_val = df[feature].median()
            df[feature].fillna(median_val, inplace=True)

print(f"Features after missing data filter: {len(features_to_keep)}")
print(features_to_keep)

In [ ]:
# Remove zero-variance features
for feature in features_to_keep[:]:
    if df[feature].nunique() <= 1:
        features_to_keep.remove(feature)
        print(f"Removed zero-variance: {feature}")

# From: scikit-learn.org (correlation analysis)
# Remove highly correlated features (Spearman |rho| > 0.85)
corr_matrix = df[features_to_keep].corr(method='spearman')
high_corr_pairs = []
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        if abs(corr_matrix.iloc[i, j]) > 0.85:
            feat1, feat2 = corr_matrix.columns[i], corr_matrix.columns[j]
            if feat2 in features_to_keep:
                features_to_keep.remove(feat2)
                print(f"Removed correlated: {feat2} (with {feat1}, rho={corr_matrix.iloc[i, j]:.3f})")

print(f"\nFinal features ({len(features_to_keep)}): {features_to_keep}")

## 3. Data Preparation

In [ ]:
X = df[features_to_keep].copy()
y = df['label'].copy()

# From: scikit-learn.org/stable/modules/preprocessing.html (standardization)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X = pd.DataFrame(X_scaled, columns=features_to_keep, index=df.index)

print(f"Standardization: mean={X.mean().mean():.6f}, std={X.std().mean():.6f}")

# Stratified train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(f"\nTrain: {X_train.shape[0]:,} | Test: {X_test.shape[0]:,}")
print(f"\nTrain distribution:\n{y_train.value_counts()}")
print(f"\nTest distribution:\n{y_test.value_counts()}")

## 4. Model Training with Hyperparameter Tuning

In [ ]:
# From: scikit-learn.org/stable/modules/ensemble.html#random-forests
# Hyperparameter grid to prevent overfitting

param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [10, 20, None],
    'min_samples_split': [5, 10],
    'min_samples_leaf': [2, 4],
    'max_features': ['sqrt', 'log2']
}

rf_base = RandomForestClassifier(
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

# From: scikit-learn.org/stable/modules/grid_search.html
grid_search = GridSearchCV(
    rf_base,
    param_grid,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring='f1_macro',
    n_jobs=-1,
    verbose=1
)

print("Starting GridSearchCV (this may take a few minutes)...")
start = time.time()
grid_search.fit(X_train, y_train)
train_time = time.time() - start

print(f"\nTraining completed in {train_time:.2f}s")
print(f"Best parameters: {grid_search.best_params_}")
print(f"Best CV F1-Score: {grid_search.best_score_:.4f}")

rf_model = grid_search.best_estimator_

## 5. Model Evaluation

In [ ]:
# Predictions
y_train_pred = rf_model.predict(X_train)
y_test_pred = rf_model.predict(X_test)

# Accuracy
train_acc = accuracy_score(y_train, y_train_pred)
test_acc = accuracy_score(y_test, y_test_pred)

print(f"Train Accuracy: {train_acc:.4f} ({train_acc*100:.2f}%)")
print(f"Test Accuracy:  {test_acc:.4f} ({test_acc*100:.2f}%)")
print(f"Overfitting Gap: {train_acc - test_acc:.4f}")

if train_acc - test_acc > 0.10:
    print("WARNING: Overfitting detected (gap > 10%)")
elif train_acc - test_acc > 0.05:
    print("NOTE: Moderate overfitting (gap 5-10%)")
else:
    print("Good generalization (gap < 5%)")

In [ ]:
# F1-Score Analysis
f1_macro = f1_score(y_test, y_test_pred, average='macro')
f1_weighted = f1_score(y_test, y_test_pred, average='weighted')

print(f"F1-Score (Macro):    {f1_macro:.4f}")
print(f"F1-Score (Weighted): {f1_weighted:.4f}")

# Per-class metrics
class_names = sorted(y.unique())
precision, recall, f1, support = precision_recall_fscore_support(
    y_test, y_test_pred, labels=class_names, zero_division=0
)

print(f"\nPer-Class Metrics:")
print(f"{'Class':<15} {'Precision':<12} {'Recall':<12} {'F1-Score':<12} {'Support':<10}")
print("-" * 65)
for i, cls in enumerate(class_names):
    print(f"{cls:<15} {precision[i]:<12.4f} {recall[i]:<12.4f} {f1[i]:<12.4f} {support[i]:<10}")

In [ ]:
# False Positive Rate Analysis
fpr_per_class = {}
for cls in class_names:
    y_true_binary = (y_test == cls).astype(int)
    y_pred_binary = (y_test_pred == cls).astype(int)
    
    tn = ((y_true_binary == 0) & (y_pred_binary == 0)).sum()
    fp = ((y_true_binary == 0) & (y_pred_binary == 1)).sum()
    
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    fpr_per_class[cls] = fpr

print(f"\nFalse Positive Rate (FPR):")
for cls, fpr in fpr_per_class.items():
    status = "GOOD" if fpr < 0.05 else "ACCEPTABLE" if fpr < 0.10 else "HIGH"
    print(f"{cls:<15}: {fpr:.4f} ({fpr*100:.2f}%) - {status}")

avg_fpr = np.mean(list(fpr_per_class.values()))
print(f"\nAverage FPR: {avg_fpr:.4f} ({avg_fpr*100:.2f}%)")

## 6. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_test_pred, labels=class_names)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix - LLDP Attack Classification', fontsize=14, fontweight='bold')
plt.ylabel('True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.tight_layout()
plt.savefig(f'{BASE_PATH}/confusion_matrix.png', dpi=300)
plt.show()

## 7. Detection Latency & Throughput

In [ ]:
# From: github.com/arsheen/IDS-on-SDN-using-Machine-Learning (latency measurement)
n_samples = 1000
sample_indices = np.random.choice(len(X_test), n_samples, replace=True)
X_latency = X_test.iloc[sample_indices]

latencies = []
for i in range(n_samples):
    sample = X_latency.iloc[i:i+1]
    start = time.perf_counter()
    _ = rf_model.predict(sample)
    latencies.append((time.perf_counter() - start) * 1000)

latencies = np.array(latencies)

print(f"Detection Latency (PacketIn to Decision):")
print(f"  Mean:   {latencies.mean():.4f} ms")
print(f"  Median: {np.median(latencies):.4f} ms")
print(f"  95th:   {np.percentile(latencies, 95):.4f} ms")
print(f"  99th:   {np.percentile(latencies, 99):.4f} ms")

# Throughput
batch_size = 1000
X_batch = X_test.iloc[:batch_size]
start = time.perf_counter()
_ = rf_model.predict(X_batch)
throughput = batch_size / (time.perf_counter() - start)

print(f"\nThroughput: {throughput:,.0f} events/second")

## 8. Feature Importance

In [ ]:
importances = rf_model.feature_importances_
importance_df = pd.DataFrame({
    'feature': features_to_keep,
    'importance': importances
}).sort_values('importance', ascending=False)

print("Feature Importance:")
print(importance_df.to_string(index=False))

plt.figure(figsize=(10, 6))
plt.barh(importance_df['feature'], importance_df['importance'], color='steelblue')
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.title('Random Forest Feature Importance')
plt.tight_layout()
plt.savefig(f'{BASE_PATH}/feature_importance.png', dpi=300)
plt.show()

## 9. Save Model & Scaler

In [ ]:
joblib.dump(rf_model, f'{BASE_PATH}/lldp_rf_model.pkl')
joblib.dump(scaler, f'{BASE_PATH}/feature_scaler.pkl')
pd.DataFrame({'feature': features_to_keep}).to_csv(f'{BASE_PATH}/feature_list.csv', index=False)

print(f"Model saved: lldp_rf_model.pkl")
print(f"Scaler saved: feature_scaler.pkl")
print(f"Features saved: feature_list.csv")

# Verify loading
loaded_model = joblib.load(f'{BASE_PATH}/lldp_rf_model.pkl')
test_pred = loaded_model.predict(X_test.iloc[:1])
print(f"\nModel loading verified (test prediction: {test_pred[0]})")

## 10. Final Summary Report

In [ ]:
print("=" * 80)
print("LLDP IDS MODEL EVALUATION SUMMARY")
print("=" * 80)
print(f"\nDataset: {len(df):,} samples, {len(class_names)} classes")
print(f"Features: {len(features_to_keep)}")
print(f"Training time: {train_time:.2f}s")
print(f"\nPerformance Metrics:")
print(f"  Test Accuracy:      {test_acc:.4f} ({test_acc*100:.2f}%)")
print(f"  F1-Score (Macro):   {f1_macro:.4f}")
print(f"  Average FPR:        {avg_fpr:.4f} ({avg_fpr*100:.2f}%)")
print(f"  Detection Latency:  {latencies.mean():.4f} ms")
print(f"  Throughput:         {throughput:,.0f} events/sec")
print(f"\nOverfitting Check:")
print(f"  Train-Test Gap:     {train_acc - test_acc:.4f}")
if train_acc - test_acc < 0.05:
    print(f"  Status: PASS (good generalization)")
else:
    print(f"  Status: WARNING (potential overfitting)")
print("\n" + "=" * 80)